In [2]:
%pip install segmentation-models-pytorch

                                              0.0/154.6 kB ? eta -:--:--
     -----------------------------------    143.4/154.6 kB 2.8 MB/s eta 0:00:01
     -------------------------------------- 154.6/154.6 kB 3.1 MB/s eta 0:00:00
                                              0.0/855.2 kB ? eta -:--:--
     --------------------------------      747.5/855.2 kB 15.7 MB/s eta 0:00:01
     ------------------------------------  849.9/855.2 kB 17.9 MB/s eta 0:00:01
     -------------------------------------- 855.2/855.2 kB 7.8 MB/s eta 0:00:00
                                              0.0/355.5 kB ? eta -:--:--
     ---------------------------------------- 355.5/355.5 kB ? eta 0:00:00
                                              0.0/2.7 MB ? eta -:--:--
     ---------------------------------------  2.7/2.7 MB 82.9 MB/s eta 0:00:01
     ---------------------------------------- 2.7/2.7 MB 56.6 MB/s eta 0:00:00
                                              0.0/125.3 kB ? eta -:--:--
    

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 23.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Imports:

import torch
from torch.utils.data import DataLoader
import os
from Utils import *
from Datasets import SegmentationDataset, TransformSubset
from Models import getModel
from FocalLoss import FocalLoss

print(torch.__version__)

c:\Users\40754\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.5.1+cu121


In [2]:
# Pre-training setup:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Set the the code to run on CUDA
print(device);

setSeed(SEED) # Set the seed for reproducibility
seededTorchGenerator = torch.Generator().manual_seed(SEED)

images = os.listdir(IMAGES_TRAIN_PATH)
labels = os.listdir(LABELS_TRAIN_PATH)

print(len(images))
print(len(labels))

classWeights = torch.tensor(WEIGHTS).to(device)

print("Creating dataset...", flush=True)
baseDataset = SegmentationDataset(
    IMAGES_TRAIN_PATH,
    LABELS_TRAIN_PATH
)
print(f"Base dataset ready. Total patches: {len(baseDataset)}", flush=True)

# 80% dataset - training data | 20% - validation data
indices = torch.randperm(len(baseDataset), generator=seededTorchGenerator).tolist()
split = int(0.8 * len(indices))
trainIndices = indices[:split]
validationIndices = indices[split:]

trainDataset = TransformSubset(
    baseDataset,
    trainIndices,
    segmentationTrainingAugmentation
)

validationDataset = TransformSubset(
    baseDataset,
    validationIndices,
    segmentationTrainingEvaluationAugmentation
)

trainLoader = DataLoader(
    trainDataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True,
    generator=seededTorchGenerator
)

validationLoader = DataLoader(
    validationDataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    generator=seededTorchGenerator
)

model = getModel(MODEL_NAME, numClasses=5).to(device)
criterion = FocalLoss(alpha=WEIGHTS, gamma=2.0, ignoreIndex=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=ADAM_LEARNING_RATE, weight_decay=1e-4)

# For early stopping in case of no improvement for the tolerance number of epochs
epochsWithoutImprovement = 0
bestValidationLoss = float('inf')
startEpoch = 0

scaler = torch.amp.GradScaler("cuda", enabled = torch.cuda.is_available()) # For mixed precision training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

bestEvaluationLoss = 0;


cuda
500
500
Creating dataset...
Total background number: 717
Cache directory:     C:\Users\40754\Desktop\Disertation\Dataset\CacheXBD
Cache files found:   492
Pre-caching complete. 246 images processed.
Cache files after:   492
Base dataset ready. Total patches: 3585


In [ ]:
# TRAINING STEP!!

# Initialize training from a saved checkpoint if specified so that the model can resume training from where it left off. This is useful for long training processes or when training is interrupted. The checkpoint contains the model's state, optimizer state, scheduler state, and the epoch number to resume from.
if RESUME_FROM is not None:
    checkpoint = torch.load(RESUME_FROM, map_location=device)
    model.load_state_dict(checkpoint["modelStateDict"])
    optimizer.load_state_dict(checkpoint["optimizerStateDict"])
    scheduler.load_state_dict(checkpoint["schedulerStateDict"])
    startEpoch = checkpoint["epoch"]
    bestValidationLoss = checkpoint["evaluationLoss"]
    print(f"Resumed training from {RESUME_FROM} at epoch {startEpoch} with val loss {bestValidationLoss:.4f}")
else:
    print("Starting training from scratch.")



for epoch in range(startEpoch, EPOCHS):
    model.train() # Set the model to training mode
    trainLoss = 0.0

    for images, masks in trainLoader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True) # Clear the gradients of all optimized tensors.

        with torch.autocast("cuda", dtype=torch.float16, enabled=True):
            outputs = model(images) # Compute the logits for the input images using the model
        loss = criterion(outputs.float(), masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        lossValue = loss.item()
        runningLoss += lossValue

    trainLoss = runningLoss / len(trainLoader)

    evaluationLoss, meanIoU = validateModel(model, validationLoader, device, epoch, criterion)

    if evaluationLoss < bestEvaluationLoss:
        bestEvaluationLoss = evaluationLoss
        epochs_without_improvement = 0
        torch.save({
            "epoch": epoch + 1,
            "modelName": MODEL_NAME,
            "modelStateDict": model.state_dict(),
            "optimizerStateDict": optimizer.state_dict(),
            "schedulerStateDict": scheduler.state_dict(),
            "trainLoss": trainLoss,
            "evaluationLoss": evaluationLoss,
            "meanIoU": meanIoU
        }, f"{MODEL_NAME}_best.pth")
        print(f"Saved best model (val loss = {evaluationLoss:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement}/{PATIENCE} epochs")
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered")
            scheduler.step()  # step before breaking so scheduler state is consistent
            break


print("\nTraining completed.")